# Corpus query, download, and review

**Date:** 2026-07-03 (system date)

This notebook helps you:

1. Search Europe PMC for scientific papers
2. Download fulltext XML where available
3. Build a scored review table for human curation
4. Refine your query and compare results

Documentation:
- [build_review_table.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/build_review_table.md)
- [aqi_india_corpus_workflow.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/aqi_india_corpus_workflow.md)
- [colab_corpus_review_notebook.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/colab_corpus_review_notebook.md)

## 1. Install dependencies

In [ ]:
!pip install -q "semantic_corpus[classification] @ git+https://github.com/semanticClimate/semantic_corpus.git@colab-review-classification"
!pip install -q pandas ipywidgets

## 2. Parameters

Edit these values before each run. Set `REVISION_OF` when refining a previous query.

In [ ]:
from pathlib import Path

QUERY_NAME = "aqi_india_pilot"
QUERY_STRING = (
    '("air quality index" OR AQI OR "ambient air pollution" OR PM2.5 OR PM10 OR "climate change") AND India'
)
LIMIT = 50
FORMATS = ["xml", "pdf"]
REPOSITORY = "europe_pmc"
NOTES = ""
REVISION_OF = None  # e.g. "aqi_india_pilot" when refining a prior run

# Classification (optional)
CLASSIFY_UNSUPERVISED = True   # cluster papers by content
N_CLUSTERS = 5
# Supervised classification against curated climate encyclopedias.
# Set to a path (e.g. Path("/content/encyclopedia")) or None to skip.
ENCYCLOPEDIA_DIR = None

OUTPUT_DIR = Path("/content/queries", QUERY_NAME)
OUTPUT_DIR

## 3. Run query, download, and build review table

In [ ]:
from semantic_corpus.corpus_review.workflow import run_query_and_build_review_table

result = run_query_and_build_review_table(
    query_name=QUERY_NAME,
    query_string=QUERY_STRING,
    output_dir=OUTPUT_DIR,
    repository=REPOSITORY,
    limit=LIMIT,
    formats=FORMATS,
    notes=NOTES,
    revision_of=REVISION_OF,
    classify_unsupervised=CLASSIFY_UNSUPERVISED,
    n_clusters=N_CLUSTERS,
    encyclopedia_dir=ENCYCLOPEDIA_DIR,
)

print(result["summary"])
print(f"Rows: {result['row_count']} (score >= 5: {result['high_score_count']}, XML: {result['xml_count']}, PDF: {result['pdf_count']})")
if result["classification"]["unsupervised"]:
    print("Clusters:")
    for cid, terms in result["classification"]["unsupervised"]["cluster_terms"].items():
        print(f"  {cid}: {', '.join(terms)}")
if result["classification"]["supervised"]:
    print(f"Encyclopedia categories: {result['classification']['supervised']['categories']}")
print("Review files:")
for fmt, path in result["review_paths"].items():
    print(f"  {fmt}: {path}")

## 4. Browse results (immediate reaction)

Inspect top candidates. Score ranks papers for review; it does **not** auto-include them.

In [ ]:
from semantic_corpus.corpus_review.workflow import review_rows_to_dataframe

df = review_rows_to_dataframe(result["rows"])

display_cols = [
    "score",
    "title",
    "pmcid",
    "has_xml",
    "has_pdf",
    "cluster_id",
    "encyclopedia_category",
    "review_status",
]
display(df[display_cols].head(15))

print("\nHigh-score papers with XML:")
high_xml = df[(df["score"] >= 5) & (df["has_xml"] == True)]
print(f"  {len(high_xml)} of {len(df)} rows")
display(high_xml[display_cols])

In [ ]:
# Optional: filter by a term in title or abstract
TERM = "Delhi"
mask = df["title"].str.contains(TERM, case=False, na=False) | df[
    "abstract_snippet"
].str.contains(TERM, case=False, na=False)
display(df.loc[mask, display_cols])

### PDF links

Papers with `has_pdf = True` have a downloadable PDF at `pdf_path`. The cell below renders clickable links and can preview a PDF inline.

In [ ]:
from IPython.display import HTML, IFrame, display as ipy_display

pdf_rows = df[df["pdf_path"] != ""]
print(f"{len(pdf_rows)} papers with PDFs")

links = []
for _, row in pdf_rows.iterrows():
    links.append(f'<li><a href="{row.pdf_path}" target="_blank">{row.title[:90]}</a></li>')
ipy_display(HTML('<ul>' + ''.join(links) + '</ul>'))

# Preview the first PDF inline (Colab serves /content files)
if len(pdf_rows):
    first_pdf = pdf_rows.iloc[0]["pdf_path"]
    print(f"Previewing: {first_pdf}")
    try:
        ipy_display(IFrame(first_pdf, width=800, height=600))
    except Exception as exc:
        print(f"Inline preview unavailable ({exc}); use the link above.")

### Classification (clusters and encyclopedia categories)

Group papers by unsupervised cluster or supervised encyclopedia category to spot themes.

In [ ]:
if "cluster_id" in df.columns and (df["cluster_id"] != "").any():
    print("Papers per cluster:")
    display(df.groupby("cluster_id")["title"].count().rename("papers"))
    CLUSTER = df["cluster_id"].iloc[0]
    print(f"\nPapers in cluster {CLUSTER}:")
    display(df[df["cluster_id"] == CLUSTER][["score", "title", "cluster_terms"]])

if "encyclopedia_category" in df.columns and (df["encyclopedia_category"] != "").any():
    print("\nPapers per encyclopedia category:")
    display(df[df["encyclopedia_category"] != ""].groupby("encyclopedia_category")["title"].count().rename("papers"))

### Optional: supervised classification with climate encyclopedias

Provide the [encyclopedia](https://github.com/semanticClimate/encyclopedia) project (curated climate wordlists) to tag papers by category. Then set `ENCYCLOPEDIA_DIR` in section 2 and re-run section 3, or run the cell below to classify the current results.

In [ ]:
# Clone the encyclopedia project (edit URL if needed), then classify current results.
# !git clone -q https://github.com/semanticClimate/encyclopedia.git /content/encyclopedia

from semantic_corpus.corpus_review.workflow import apply_classification
from semantic_corpus.corpus_review.review_table import export_review_tables

ENCYCLOPEDIA_DIR = Path("/content/encyclopedia")  # adjust to your checkout
if ENCYCLOPEDIA_DIR.is_dir():
    apply_classification(
        result["rows"],
        search_results_path=result["search_results_path"],
        classify_unsupervised=True,
        n_clusters=N_CLUSTERS,
        encyclopedia_dir=ENCYCLOPEDIA_DIR,
    )
    export_review_tables(result["rows"], Path(OUTPUT_DIR, "review"))
    df = review_rows_to_dataframe(result["rows"])
    print("Reclassified; review tables updated.")
else:
    print(f"Encyclopedia not found at {ENCYCLOPEDIA_DIR}; clone it first.")

## 5. Download review artifacts

Download the CSV or JSON to edit `review_status` (`include`, `review`, `exclude`) offline.

In [ ]:
from google.colab import files
import shutil

csv_path = result["review_paths"]["csv"]
files.download(csv_path)

# Optional: zip the entire query directory
zip_base = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
files.download(zip_base)

## 6. Iterative query refinement

Use this checklist after each run:

- **Too broad?** Add location or topic terms (e.g. `AND Delhi`)
- **Too narrow?** Remove restrictive terms or broaden synonyms
- **Missing fulltext?** Papers without PMCID cannot download XML; note `has_xml = false` rows

To refine: copy parameters into section 2, set `REVISION_OF` to the previous `QUERY_NAME`, and re-run sections 3–5.

In [ ]:
# Example: refined query (edit QUERY_NAME and QUERY_STRING in section 2, then re-run)
REFINED_QUERY_NAME = "aqi_india_delhi"
REFINED_QUERY_STRING = (
    '("air quality index" OR AQI OR PM2.5) AND India AND Delhi'
)
print("Set in section 2:")
print(f"  QUERY_NAME = {REFINED_QUERY_NAME!r}")
print(f"  QUERY_STRING = {REFINED_QUERY_STRING!r}")
print(f"  REVISION_OF = {QUERY_NAME!r}")

In [ ]:
# Compare two runs by pmcid/pmid (run after a refined query)
import json

def load_ids(path):
    with open(path, "r", encoding="utf-8") as handle:
        rows = json.load(handle)
    ids = set()
    for row in rows:
        key = row.get("pmcid") or row.get("pmid") or row.get("paper_id")
        if key:
            ids.add(key)
    return ids

prior_json = Path("/content/queries", QUERY_NAME if REVISION_OF is None else REVISION_OF, "review", "review_table.json")
current_json = Path(result["review_paths"]["json"])

if prior_json.is_file() and current_json.is_file():
    prior_ids = load_ids(prior_json)
    current_ids = load_ids(current_json)
    overlap = prior_ids & current_ids
    print(f"Prior run: {len(prior_ids)} papers")
    print(f"Current run: {len(current_ids)} papers")
    print(f"Overlap: {len(overlap)} papers")
    print(f"New in current run: {len(current_ids - prior_ids)}")
    print(f"Dropped from prior run: {len(prior_ids - current_ids)}")
else:
    print("Run a refined query first, or adjust prior_json path.")

## 7. Optional: persist to Google Drive

Colab sessions are ephemeral. Mount Drive and set `OUTPUT_DIR` under your Drive folder to keep results between sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Example: OUTPUT_DIR = Path("/content/drive/MyDrive/semantic_corpus/queries", QUERY_NAME)

## 8. Next steps (outside this notebook)

1. Edit `review_status` in the downloaded CSV or JSON (`include` / `exclude`)
2. Ingest selected papers into a BAGIT corpus locally
3. Export a chatbot manifest for RAG indexing

See the [chatbot export contract](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/chatbot_export_contract.md).